# Generalized full-frame paper-theater notebook

This version keeps the original pipeline structure, but removes the most fragile scene-specific pieces.

## What changed
- automatic scene naming from the input image
- optional Gemini / OpenAI vision scene analysis
- dynamic Florence query generation instead of a fixed object list
- generalized label normalization into semantic families
- generalized fallback-mask restriction by semantic family instead of exact labels
- generalized layer prompts based on layer role and front/behind context
- multiple debugging cells to inspect:
  - scene analysis JSON
  - generated Florence queries
  - detection overlays
  - segmentation masks
  - depth map
  - layer summaries
  - focus masks sent to the image model

## Recommended use
1. Put one input image in the `data/input` folder.
2. Set `IMAGE_PATH` in the config cell.
3. Run the notebook top to bottom.
4. Inspect the debug cells before generating the final full-frame layers.

## Patch notes

This patched version keeps the AI scene-analysis step, but stabilizes the generalized pipeline by:
- constraining Florence queries to broad canonical object families
- normalizing labels before downstream grouping
- dropping explicit ground/detail detections such as path, grass, and bag
- reordering final layer contexts with depth-aware postprocessing
- keeping fallback pixels broad and preventing them from flooding subject layers
- returning to an anchor-first show mask closer to the stable scene2 behavior


In [ ]:
%cd /content
!git clone https://github.com/UrbinaDan/PaperTheater_ai_Pipeline.git
%cd /content/PaperTheater_ai_Pipeline

!pip install -q numpy scipy matplotlib opencv-python-headless pillow shapely svgwrite cairosvg tqdm networkx pyyaml requests google-genai openai accelerate bitsandbytes einops sentencepiece transformers==4.49.0 pandas

%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd /content/PaperTheater_ai_Pipeline

In [ ]:
import os
from pathlib import Path
from getpass import getpass

# =========================
# USER CONFIG
# =========================

IMAGE_PATH = "/content/PaperTheater_ai_Pipeline/data/input/scene_2.jpg"
VISION_PROVIDER = "gemini"           # "gemini", "openai", or "none"
IMAGE_GENERATION_PROVIDER = "gemini" # final generation currently uses Gemini
USE_VISION_LLM = True
USE_LLM_LAYER_GUIDANCE = True

GEMINI_TEXT_MODEL = "gemini-2.5-flash"
GEMINI_IMAGE_MODEL = "gemini-3.1-flash-image-preview"
OPENAI_TEXT_MODEL = "gpt-5"

MAX_QUERIES = 8
DEBUG_SHOW_EACH_OBJECT_MASK = True
DEBUG_SHOW_EACH_LAYER_MASK = True

STYLE_NORMALIZATION_PROMPT = """
Render this as a stylized Japanese paper theater / kamishibai layer.

GLOBAL STYLE RULES:
- flat paper-cut illustration
- simplified silhouettes and clean edges
- muted earthy palette
- minimal texture
- no photorealism
- no realistic skin or fur rendering
- no detailed lighting or harsh shadows
- consistent abstraction across all layers
- all layers should look like they were made by the same artist
- preserve the original composition and scale inside the full canvas
- output should feel like a clean cuttable layer for paper theater production
""".strip()

if USE_VISION_LLM and VISION_PROVIDER == "gemini" and "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

if USE_VISION_LLM and VISION_PROVIDER == "openai" and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if IMAGE_GENERATION_PROVIDER == "gemini" and "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key for image generation: ")
USE_RAW_DEPTH_FALLBACK_FOR_GEMINI = True
DEBUG_SHOW_LEFTOVER_ASSIGNMENT = True


In [ ]:
import os
import io
import re
import ast
import json
import time
import base64
import shutil
import textwrap
import importlib
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from google import genai
from google.genai.errors import ServerError, ClientError
from openai import OpenAI

from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image, save_mask
from src.florence_parser import FlorenceParser
from src.sam2_segmenter import SAM2Segmenter
from src.depth_anything_runner import DepthRunner
from src.occlusion_heuristic import heuristic_complete
from src.mask_cleanup import cleanup_mask

import src.scene_builder
importlib.reload(src.scene_builder)
from src.scene_builder import (
    parse_florence_boxes,
    merge_segmented_by_label,
    build_stable_merged_objects,
)

import src.scene_representation
importlib.reload(src.scene_representation)
from src.scene_representation import build_scene_representation, save_scene_representation

import src.layer_planner
importlib.reload(src.layer_planner)
from src.layer_planner import plan_layers_deterministic

import src.layer_renderer
importlib.reload(src.layer_renderer)
from src.layer_renderer import build_object_mask_map

import src.layer_context_builder
importlib.reload(src.layer_context_builder)
from src.layer_context_builder import build_layer_contexts

paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"]) if os.environ.get("GEMINI_API_KEY") else None
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"]) if os.environ.get("OPENAI_API_KEY") else None

In [ ]:
# =========================
# PATHS + DERIVED CONFIG
# =========================

image_path = IMAGE_PATH
SCENE_NAME = Path(image_path).stem

BASE_INTERMEDIATE_DIR = Path("/content/PaperTheater_ai_Pipeline/data/intermediate")
SCENE_DIR = BASE_INTERMEDIATE_DIR / SCENE_NAME
FULLFRAME_OUTPUT_DIR = SCENE_DIR / "fullframe_outputs"
DEBUG_DIR = SCENE_DIR / "debug"
MASK_DIR = SCENE_DIR / "masks"

SCENE_REPR_PATH = SCENE_DIR / f"{SCENE_NAME}_scene_repr.json"
LAYER_PLAN_PATH = SCENE_DIR / f"{SCENE_NAME}_layer_plan.json"
SCENE_ANALYSIS_PATH = SCENE_DIR / f"{SCENE_NAME}_scene_analysis.json"
QUERY_DEBUG_PATH = SCENE_DIR / f"{SCENE_NAME}_queries.json"
LAYER_GUIDANCE_PATH = SCENE_DIR / f"{SCENE_NAME}_layer_guidance.json"

for p in [SCENE_DIR, FULLFRAME_OUTPUT_DIR, DEBUG_DIR, MASK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("SCENE_NAME:", SCENE_NAME)
print("image_path:", image_path)
print("SCENE_DIR:", SCENE_DIR)
print("FULLFRAME_OUTPUT_DIR:", FULLFRAME_OUTPUT_DIR)

In [ ]:
# =========================
# HELPERS
# =========================

def show_image(img, title="", figsize=(8, 12), cmap=None):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

def image_to_pil(image_rgb):
    if isinstance(image_rgb, Image.Image):
        return image_rgb.convert("RGB")
    if isinstance(image_rgb, np.ndarray):
        if image_rgb.ndim == 2:
            image_rgb = np.stack([image_rgb] * 3, axis=-1)
        if image_rgb.shape[-1] == 4:
            image_rgb = image_rgb[..., :3]
        return Image.fromarray(image_rgb.astype(np.uint8)).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(image_rgb)}")

def normalize_to_rgb_array(img_obj):
    if isinstance(img_obj, np.ndarray):
        arr = img_obj
        if arr.ndim == 2:
            return np.stack([arr] * 3, axis=-1).astype(np.uint8)
        if arr.shape[-1] == 4:
            return arr[..., :3].astype(np.uint8)
        return arr.astype(np.uint8)
    if isinstance(img_obj, Image.Image):
        return np.array(img_obj.convert("RGB"))
    if isinstance(img_obj, (bytes, bytearray)):
        return np.array(Image.open(io.BytesIO(img_obj)).convert("RGB"))
    if hasattr(img_obj, "image_bytes") and img_obj.image_bytes is not None:
        raw = img_obj.image_bytes
        if isinstance(raw, str):
            raw = base64.b64decode(raw)
        return np.array(Image.open(io.BytesIO(raw)).convert("RGB"))
    if hasattr(img_obj, "numpy_view"):
        arr = np.array(img_obj.numpy_view())
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)
        elif arr.shape[-1] == 4:
            arr = arr[..., :3]
        return arr.astype(np.uint8)
    raise TypeError(f"Unsupported image object type: {type(img_obj)}")

def save_bool_mask(mask, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)

def mask_to_rgba_fullframe(image_rgb, mask):
    alpha = (mask.astype(np.uint8)) * 255
    return np.dstack([image_rgb, alpha])

def safe_json_loads(text, default=None):
    if default is None:
        default = {}
    if text is None:
        return default
    text = text.strip()
    if not text:
        return default
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except Exception:
        pass
    match_obj = re.search(r"\{.*\}", text, flags=re.DOTALL)
    match_arr = re.search(r"\[.*\]", text, flags=re.DOTALL)
    candidates = []
    if match_obj:
        candidates.append(match_obj.group(0))
    if match_arr:
        candidates.append(match_arr.group(0))
    for cand in candidates:
        try:
            return json.loads(cand)
        except Exception:
            continue
    for cand in [text] + candidates:
        try:
            return ast.literal_eval(cand)
        except Exception:
            continue
    return default

def clean_query_text(q):
    q = str(q).strip().lower()
    q = re.sub(r"^[\-*\d\.\)\s]+", "", q)
    q = q.replace("_", " ")
    q = re.sub(r"\s+", " ", q).strip()
    q = q[:60]
    return q

def dedupe_keep_order(items):
    out = []
    seen = set()
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def clamp_box(bbox, image_shape):
    h, w = image_shape[:2]
    x1, y1, x2, y2 = bbox
    x1 = int(max(0, min(w - 1, x1)))
    x2 = int(max(0, min(w - 1, x2)))
    y1 = int(max(0, min(h - 1, y1)))
    y2 = int(max(0, min(h - 1, y2)))
    if x2 <= x1:
        x2 = min(w - 1, x1 + 1)
    if y2 <= y1:
        y2 = min(h - 1, y1 + 1)
    return [x1, y1, x2, y2]

def box_iou(boxA, boxB):
    ax1, ay1, ax2, ay2 = boxA
    bx1, by1, bx2, by2 = boxB
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    areaA = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    areaB = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

def dedupe_boxes(boxes, image_shape, iou_threshold=0.8):
    clean = []
    for b in boxes:
        label = clean_query_text(b.get("label", "object"))
        bbox = clamp_box(b["bbox"], image_shape)
        keep = True
        for prev in clean:
            if prev["label"] == label and box_iou(prev["bbox"], bbox) >= iou_threshold:
                keep = False
                break
        if keep:
            clean.append({"label": label, "bbox": bbox})
    return clean

def area_fraction(mask):
    return float(mask.sum()) / float(mask.size) if mask.size else 0.0

def dynamic_kernel_from_mask(mask, min_k=9, max_k=151, frac=0.08):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return min_k
    bw = xs.max() - xs.min() + 1
    bh = ys.max() - ys.min() + 1
    ref = max(bw, bh)
    k = int(max(min_k, min(max_k, ref * frac)))
    if k % 2 == 0:
        k += 1
    return k

def expand_mask(mask, kernel_size=None):
    if kernel_size is None:
        kernel_size = dynamic_kernel_from_mask(mask.astype(np.uint8))
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    expanded = cv2.dilate(mask.astype(np.uint8), kernel, iterations=1)
    return expanded > 0

def normalize_label_family(label):
    l = clean_query_text(label)
    family_rules = [
        ("sky_background", ["sky", "cloud", "sun", "moon"]),
        ("distant_landscape", ["mountain", "hill", "horizon", "background mountain", "background hills"]),
        ("architecture", ["temple", "pagoda", "church", "building", "house", "tower", "shrine", "roof", "gate"]),
        ("vegetation", ["tree", "forest", "bush", "plant", "flower", "blossom", "branch", "leaf", "grass"]),
        ("human_subject", ["person", "man", "woman", "boy", "girl", "people", "tourist", "child"]),
        ("animal_subject", ["deer", "dog", "cat", "animal", "horse", "bird"]),
        ("ground_plane", ["ground", "path", "road", "walkway", "floor", "field", "lawn", "grassland"]),
        ("manmade_object", ["statue", "sign", "poster", "luggage", "suitcase", "bag", "bench", "car", "fence"]),
    ]
    for family, kws in family_rules:
        if any(kw in l for kw in kws):
            return family
    return "object"

def get_role_guidance(role):
    guidance = {
        "sky_background": [
            "This layer must contain only sky-like content.",
            "Do not include land, trees, buildings, people, animals, or ground.",
            "Beige regions should be completed only with sky continuation or very soft cloud continuation."
        ],
        "distant_landscape": [
            "Complete the distant landscape only.",
            "Keep it broad, quiet, and low-detail.",
            "Do not add foreground objects or strong texture."
        ],
        "architecture": [
            "Complete only the architecture and simple support shapes around it.",
            "Preserve structural alignment and silhouette logic.",
            "Do not invent a new building or add extra people or animals."
        ],
        "vegetation": [
            "Complete the vegetation mass with simplified foliage continuation.",
            "If beige gaps appear behind removed front objects, continue the foliage softly and cleanly.",
            "Do not add new focal subjects."
        ],
        "human_subject": [
            "Keep the person as the main subject for this layer.",
            "Fill beige regions only with minimal support continuation appropriate to the layer.",
            "Do not add extra people or animals."
        ],
        "animal_subject": [
            "Keep the animal as the main subject for this layer.",
            "Complete surrounding support shapes simply and quietly.",
            "Do not add extra animals or people."
        ],
        "ground_plane": [
            "Complete the ground plane as broad flat support shapes.",
            "Keep it simple and low-detail.",
            "Do not add focal objects."
        ],
        "manmade_object": [
            "Complete only the target object and very simple support continuation.",
            "Do not invent extra unrelated props.",
            "Preserve scene alignment."
        ],
        "object": [
            "Complete only the current layer.",
            "Use broad flat continuation.",
            "Do not invent unrelated focal objects."
        ],
    }
    return guidance.get(role, guidance["object"])

def compute_layer_depth_stats(layer_contexts, depth_map):
    stats = {}
    global_depth = float(np.median(depth_map))
    for ctx in layer_contexts:
        ownership_mask = ctx["ownership_mask"] > 0
        vals = depth_map[ownership_mask]
        rep_depth = float(np.median(vals)) if vals.size else global_depth
        stats[ctx["layer_name"]] = {"order": ctx["order"], "rep_depth": rep_depth}
    return stats

def build_unassigned_pixel_mask(layer_contexts, image_shape):
    H, W = image_shape[:2]
    assigned = np.zeros((H, W), dtype=bool)
    for ctx in layer_contexts:
        assigned |= (ctx["ownership_mask"] > 0)
    return ~assigned


def assign_leftover_pixels_by_depth(layer_contexts, depth_map):
    H, W = depth_map.shape
    layer_stats = compute_layer_depth_stats(layer_contexts, depth_map)
    unassigned = build_unassigned_pixel_mask(layer_contexts, (H, W))
    fallback_masks = {ctx["layer_name"]: np.zeros((H, W), dtype=bool) for ctx in layer_contexts}
    assignment_index = np.full((H, W), -1, dtype=np.int32)

    layer_names = [ctx["layer_name"] for ctx in layer_contexts]
    layer_depths = np.array([layer_stats[name]["rep_depth"] for name in layer_names], dtype=np.float32)

    ys, xs = np.where(unassigned)
    if len(ys) == 0:
        return fallback_masks, unassigned, layer_stats, assignment_index

    pixel_depths = depth_map[ys, xs].astype(np.float32)
    dist = np.abs(pixel_depths[:, None] - layer_depths[None, :])
    best_idx = np.argmin(dist, axis=1)

    for k, layer_idx in enumerate(best_idx):
        y = ys[k]
        x = xs[k]
        fallback_masks[layer_names[layer_idx]][y, x] = True
        assignment_index[y, x] = int(layer_idx)

    return fallback_masks, unassigned, layer_stats, assignment_index

def restrict_fallback_masks_generalized(layer_contexts, depth_fallback_masks):
    restricted = {}
    for ctx in layer_contexts:
        layer_name = ctx["layer_name"]
        fallback_mask = depth_fallback_masks[layer_name] > 0
        ownership_mask = ctx["ownership_mask"] > 0
        role = ctx.get("family", "object")
        h, w = ownership_mask.shape

        if role in {"human_subject", "animal_subject", "manmade_object"}:
            local_window = expand_mask(ownership_mask.astype(np.uint8), kernel_size=dynamic_kernel_from_mask(ownership_mask, min_k=31, max_k=181, frac=0.18))
            fallback_mask = fallback_mask & local_window
        elif role == "architecture":
            local_window = expand_mask(ownership_mask.astype(np.uint8), kernel_size=dynamic_kernel_from_mask(ownership_mask, min_k=41, max_k=221, frac=0.24))
            fallback_mask = fallback_mask & local_window
        elif role == "ground_plane":
            yy = np.arange(h)[:, None]
            lower_half = yy >= int(h * 0.35)
            fallback_mask = fallback_mask & lower_half
        elif role == "sky_background":
            yy = np.arange(h)[:, None]
            upper_zone = yy <= int(h * 0.7)
            fallback_mask = fallback_mask & upper_zone

        restricted[layer_name] = fallback_mask
    return restricted


def summarize_leftover_assignment(layer_contexts, raw_depth_fallback_masks, restricted_depth_fallback_masks, unassigned_pixels_mask, assignment_index):
    print("\nLeftover pixel assignment summary:")
    total_leftover = int(unassigned_pixels_mask.sum())
    print("total leftover pixels:", total_leftover)
    if total_leftover == 0:
        return

    for idx_layer, ctx in enumerate(layer_contexts):
        layer_name = ctx["layer_name"]
        raw_count = int(raw_depth_fallback_masks[layer_name].sum())
        restricted_count = int(restricted_depth_fallback_masks[layer_name].sum())
        assigned_count = int((assignment_index == idx_layer).sum())
        print({
            "layer_name": layer_name,
            "raw_depth_assigned": raw_count,
            "restricted_depth_assigned": restricted_count,
            "assignment_index_count": assigned_count,
        })

def show_leftover_assignment_map(layer_contexts, image_rgb, unassigned_pixels_mask, assignment_index):
    h, w = unassigned_pixels_mask.shape
    color_map = np.zeros((h, w, 3), dtype=np.uint8)
    rng = np.random.default_rng(0)
    palette = rng.integers(40, 255, size=(max(1, len(layer_contexts)), 3), dtype=np.uint8)

    for idx_layer, _ctx in enumerate(layer_contexts):
        color_map[assignment_index == idx_layer] = palette[idx_layer]

    fig, axes = plt.subplots(1, 3, figsize=(18, 8))

    axes[0].imshow(image_rgb)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    axes[1].imshow(unassigned_pixels_mask, cmap="gray")
    axes[1].set_title("Leftover pixels")
    axes[1].axis("off")

    axes[2].imshow(color_map)
    axes[2].set_title("Leftover -> nearest depth layer")
    axes[2].axis("off")

    plt.show()

def get_layer_support_mask(layer_context, depth_fallback_masks):
    visible_mask = layer_context["visible_mask"] > 0
    fallback_mask = depth_fallback_masks[layer_context["layer_name"]] > 0
    return visible_mask | fallback_mask

def build_control_masks_for_layer(layer_context, ordered_contexts, depth_fallback_masks):
    order = layer_context["order"]
    layer_name = layer_context["layer_name"]
    role = layer_context.get("family", "object")
    visible_mask = layer_context["visible_mask"] > 0
    fallback_mask = depth_fallback_masks[layer_name] > 0
    current_support = visible_mask | fallback_mask
    H, W = visible_mask.shape
    front_visible = np.zeros((H, W), dtype=bool)
    behind_support = np.zeros((H, W), dtype=bool)

    for ctx in ordered_contexts:
        if ctx["layer_name"] == layer_name:
            continue
        if ctx["order"] > order:
            front_visible |= (ctx["visible_mask"] > 0)
        elif ctx["order"] < order:
            behind_support |= get_layer_support_mask(ctx, depth_fallback_masks)

    if role == "sky_background":
        show_mask = current_support
        black_mask = np.zeros((H, W), dtype=bool)
    else:
        show_mask = current_support
        black_mask = behind_support.copy()

    beige_mask = ~(show_mask | black_mask)

    return {
        "anchor_mask": visible_mask,
        "fallback_mask": fallback_mask,
        "current_support": current_support,
        "front_visible": front_visible,
        "behind_support": behind_support,
        "show_mask": show_mask,
        "black_mask": black_mask,
        "beige_mask": beige_mask,
    }

def apply_focus_mask_fullframe(image_rgb, show_mask, black_mask, beige_value=(232, 224, 210)):
    h, w = show_mask.shape
    out = np.full((h, w, 3), beige_value, dtype=np.uint8)
    out[black_mask > 0] = 0
    out[show_mask > 0] = image_rgb[show_mask > 0]
    return out

def show_mask_debug(
    image_rgb,
    layer_name,
    anchor_mask,
    fallback_mask,
    current_support,
    front_visible,
    behind_support,
    show_mask,
    black_mask,
    focused_input,
):
    h, w = anchor_mask.shape
    comp = np.zeros((h, w, 3), dtype=np.uint8)
    comp[anchor_mask] = [80, 140, 255]
    comp[fallback_mask] = [220, 80, 220]
    comp[front_visible] = [80, 220, 220]
    comp[behind_support] = [220, 80, 80]

    fig, axes = plt.subplots(2, 3, figsize=(16, 18))
    axes = axes.ravel()

    axes[0].imshow(image_rgb)
    axes[0].set_title(f"{layer_name} - original")
    axes[0].axis("off")

    axes[1].imshow(current_support, cmap="gray")
    axes[1].set_title("current_support")
    axes[1].axis("off")

    axes[2].imshow(front_visible, cmap="gray")
    axes[2].set_title("front_visible")
    axes[2].axis("off")

    axes[3].imshow(behind_support, cmap="gray")
    axes[3].set_title("behind_support (black)")
    axes[3].axis("off")

    axes[4].imshow(comp)
    axes[4].set_title("blue=anchor magenta=fallback cyan=front red=behind support")
    axes[4].axis("off")

    axes[5].imshow(focused_input)
    axes[5].set_title("focused_input sent to the image model")
    axes[5].axis("off")

    plt.tight_layout()
    plt.show()

def call_vision_text_llm(image_rgb, prompt, provider="gemini"):
    pil_img = image_to_pil(image_rgb)

    if provider == "gemini":
        if gemini_client is None:
            raise RuntimeError("Gemini client is not available.")
        response = gemini_client.models.generate_content(
            model=GEMINI_TEXT_MODEL,
            contents=[prompt, pil_img],
        )
        return getattr(response, "text", "") or ""

    if provider == "openai":
        if openai_client is None:
            raise RuntimeError("OpenAI client is not available.")
        buffered = io.BytesIO()
        pil_img.save(buffered, format="PNG")
        b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
        response = openai_client.responses.create(
            model=OPENAI_TEXT_MODEL,
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": f"data:image/png;base64,{b64}"},
                ],
            }],
        )
        return getattr(response, "output_text", "") or ""

    raise ValueError(f"Unsupported provider: {provider}")

def analyze_scene_with_llm(image_rgb, fallback_caption="", provider="gemini"):
    prompt = f"""
Analyze this image for a paper-theater layer extraction pipeline.

Return STRICT JSON only with this exact schema:
{{
  "scene_summary": "1-3 sentence summary",
  "style_tags": ["tag1", "tag2"],
  "important_objects": ["object1", "object2"],
  "framing_elements": ["element1", "element2"],
  "background_elements": ["element1", "element2"],
  "foreground_elements": ["element1", "element2"],
  "candidate_detection_queries": ["query1", "query2"],
  "likely_indoor_or_outdoor": "indoor|outdoor|mixed",
  "notes_for_layering": ["note1", "note2"]
}}

Rules:
- Keep candidate_detection_queries short and concrete.
- Prefer nouns that Florence could detect.
- Include only visually important objects.
- Do not add explanations outside the JSON.

Fallback caption:
{fallback_caption}
""".strip()

    raw = call_vision_text_llm(image_rgb, prompt, provider=provider)
    parsed = safe_json_loads(raw, default={})
    if not isinstance(parsed, dict):
        parsed = {}
    parsed.setdefault("scene_summary", fallback_caption or "")
    parsed.setdefault("style_tags", [])
    parsed.setdefault("important_objects", [])
    parsed.setdefault("framing_elements", [])
    parsed.setdefault("background_elements", [])
    parsed.setdefault("foreground_elements", [])
    parsed.setdefault("candidate_detection_queries", [])
    parsed.setdefault("likely_indoor_or_outdoor", "mixed")
    parsed.setdefault("notes_for_layering", [])
    return parsed

def generate_queries(scene_analysis, fallback_caption="", max_queries=12):
    queries = []
    if isinstance(scene_analysis, dict):
        queries.extend(scene_analysis.get("candidate_detection_queries", []))
        queries.extend(scene_analysis.get("important_objects", []))
        queries.extend(scene_analysis.get("background_elements", []))
        queries.extend(scene_analysis.get("framing_elements", []))
        queries.extend(scene_analysis.get("foreground_elements", []))

    backup_queries = [
        "person", "people", "animal", "deer", "dog", "cat",
        "tree", "forest", "flower", "blossom", "branch", "leaf",
        "building", "temple", "pagoda", "church", "house", "tower",
        "mountain", "hill", "sky", "cloud",
        "path", "road", "ground", "field", "grass",
        "statue", "poster", "sign", "luggage", "bag", "bench"
    ]
    cap_lower = (fallback_caption or "").lower()
    for q in backup_queries:
        if q in cap_lower:
            queries.append(q)

    queries.extend(["sky", "tree", "building", "person"])

    queries = [clean_query_text(q) for q in queries]
    queries = [q for q in queries if q and len(q) >= 2]
    queries = dedupe_keep_order(queries)
    return queries[:max_queries]

def summarize_layer_contexts(layer_contexts):
    rows = []
    for ctx in sorted(layer_contexts, key=lambda x: x["order"]):
        rows.append({
            "order": ctx["order"],
            "layer_name": ctx["layer_name"],
            "family": ctx.get("family", ""),
            "labels": ", ".join(ctx.get("labels", [])),
            "object_ids": ", ".join(map(str, ctx.get("object_ids", []))),
            "front_layers": ", ".join(ctx.get("front_layer_names", [])),
            "visible_area_frac": round(area_fraction(ctx["visible_mask"] > 0), 4),
        })
    return pd.DataFrame(rows)

def build_layer_guidance_fallback(layer_context, scene_analysis):
    labels = layer_context.get("labels", [])
    role = layer_context.get("family", "object")
    front_layer_labels = layer_context.get("front_layer_labels", [])
    forbidden = dedupe_keep_order(front_layer_labels)
    return {
        "layer_goal": f"complete the {layer_context['layer_name']} layer as a single full-frame paper-theater layer",
        "must_include": labels[:3],
        "allowed_extensions": scene_analysis.get("background_elements", [])[:3] if isinstance(scene_analysis, dict) else [],
        "forbidden_elements": forbidden[:8],
        "count_guidance": "Do not add extra repeated focal objects.",
        "density_guidance": "Keep filler broad, flat, simple, and low-detail.",
        "placement_guidance": "Preserve alignment with the anchor pixels and complete only the current layer.",
        "style_guidance": "Keep the same stylized paper-theater look as the source image.",
    }

def build_layer_plan_request(layer_context, scene_analysis):
    labels = layer_context.get("labels", [])
    primary_label = labels[0] if labels else layer_context["layer_name"]
    return {
        "scene_summary": scene_analysis.get("scene_summary", ""),
        "scene_style_tags": scene_analysis.get("style_tags", []),
        "layer_name": layer_context["layer_name"],
        "layer_family": layer_context.get("family", "object"),
        "primary_label": primary_label,
        "labels": labels,
        "front_layer_names": layer_context.get("front_layer_names", []),
        "front_layer_labels": layer_context.get("front_layer_labels", []),
        "background_elements": scene_analysis.get("background_elements", []),
        "foreground_elements": scene_analysis.get("foreground_elements", []),
        "framing_elements": scene_analysis.get("framing_elements", []),
    }

def build_layer_plan_prompt(layer_context, scene_analysis):
    payload = build_layer_plan_request(layer_context, scene_analysis)
    return f"""
You are planning one image layer for a paper-theater generation pipeline.

Return STRICT JSON only with this exact schema:
{{
  "layer_goal": "short phrase",
  "must_include": ["item1", "item2"],
  "allowed_extensions": ["item1", "item2"],
  "forbidden_elements": ["item1", "item2"],
  "count_guidance": "short sentence",
  "density_guidance": "short sentence",
  "placement_guidance": "short sentence",
  "style_guidance": "short sentence"
}}

Rules:
- Focus on what should appear in THIS layer only.
- forbidden_elements should mostly be front-layer content or unrelated focal objects.
- Be concise.
- Output only JSON.

Layer context JSON:
{json.dumps(payload, ensure_ascii=False, indent=2)}
""".strip()

def get_layer_guidance_with_llm(layer_context, scene_analysis, provider="gemini"):
    prompt = build_layer_plan_prompt(layer_context, scene_analysis)
    raw = call_vision_text_llm(image, prompt, provider=provider)
    parsed = safe_json_loads(raw, default={})
    if not isinstance(parsed, dict) or "layer_goal" not in parsed:
        parsed = build_layer_guidance_fallback(layer_context, scene_analysis)
    return parsed

def build_fullframe_prompt(layer_context, layer_guidance):
    label = layer_context["labels"][0] if layer_context["labels"] else layer_context["layer_name"]
    layer_name = layer_context["layer_name"]
    role = layer_context.get("family", "object")
    front_layer_labels = dedupe_keep_order(layer_context.get("front_layer_labels", []))
    role_guidance = get_role_guidance(role)

    role_guidance_text = "\n".join([f"- {x}" for x in role_guidance])
    must_include_text = "\n".join([f"- {x}" for x in layer_guidance.get("must_include", [])]) or "- preserve the current target layer content"
    allowed_extensions_text = "\n".join([f"- {x}" for x in layer_guidance.get("allowed_extensions", [])]) or "- simple continuation only"
    forbidden_text = "\n".join([f"- {x}" for x in dedupe_keep_order(layer_guidance.get("forbidden_elements", []) + front_layer_labels)]) or "- unrelated objects from other layers"

    return f"""
Generate a FULL-FRAME layer for a layered paper-theater scene.

Scene context:
{layer_context.get("scene_caption", "")}

Target layer:
- layer name: {layer_name}
- semantic label: {label}
- semantic family: {role}

{STYLE_NORMALIZATION_PROMPT}

INPUT INTERPRETATION:
- Colored visible pixels belong to the current target layer and must be preserved in alignment and style.
- Black regions are forbidden. Do not paint into black regions.
- Beige regions are open canvas that must be used to COMPLETE the current layer.
- The final output should look like one finished paper-theater layer, not a partial cutout floating on blank paper.

ROLE-SPECIFIC GUIDANCE:
{role_guidance_text}

LAYER GOAL:
- {layer_guidance.get("layer_goal", "complete the current layer cleanly")}

MUST INCLUDE:
{must_include_text}

ALLOWED EXTENSIONS:
{allowed_extensions_text}

FORBIDDEN ELEMENTS:
{forbidden_text}

ADDITIONAL GUIDANCE:
- {layer_guidance.get("count_guidance", "Do not add extra focal objects.")}
- {layer_guidance.get("density_guidance", "Keep filler broad and low-detail.")}
- {layer_guidance.get("placement_guidance", "Preserve exact scene alignment.")}
- {layer_guidance.get("style_guidance", "Keep a consistent paper-theater style.")}

IMPORTANT RULES:
1. Generate only the current target layer.
2. Preserve exact scene alignment with the original image.
3. Do not invent unrelated objects from other layers.
4. Do not paint into black regions.
5. Fill beige regions so the layer feels complete.
6. Keep filler simple and low-detail.
7. The result should look like a finished cuttable paper-theater layer.
8. If a front object was masked out, either reconstruct what belongs behind it or remove the gap cleanly.
9. Do not leave large blank beige holes.

OUTPUT GOAL:
A complete full-frame paper-theater layer that preserves the visible anchor, respects black forbidden regions, and fills beige regions with appropriate continuation.
""".strip()

def gemini_edit(image_rgb, prompt, model_name, max_retries=4):
    if gemini_client is None:
        raise RuntimeError("Gemini client is not available.")
    pil_img = image_to_pil(image_rgb)
    last_err = None
    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model_name,
                contents=[prompt, pil_img],
            )

            if hasattr(response, "generated_images") and response.generated_images:
                img_obj = response.generated_images[0].image
                return normalize_to_rgb_array(img_obj)

            if hasattr(response, "candidates") and response.candidates:
                for cand in response.candidates:
                    content = getattr(cand, "content", None)
                    if content is None:
                        continue
                    for part in getattr(content, "parts", []):
                        inline_data = getattr(part, "inline_data", None)
                        if inline_data is not None and getattr(inline_data, "data", None):
                            raw = inline_data.data
                            if isinstance(raw, str):
                                raw = base64.b64decode(raw)
                            return np.array(Image.open(io.BytesIO(raw)).convert("RGB"))
                        if hasattr(part, "as_image"):
                            maybe_img = part.as_image()
                            return normalize_to_rgb_array(maybe_img)

            if hasattr(response, "text"):
                print("No usable image found. Gemini text preview:", repr(getattr(response, "text", None))[:800])
            raise ValueError("Gemini response did not contain a usable image")

        except ServerError as e:
            last_err = e
            wait_s = min(60, 5 * (2 ** attempt))
            print(f"Gemini ServerError on attempt {attempt+1}/{max_retries}. Retrying in {wait_s}s...")
            time.sleep(wait_s)
        except ClientError:
            raise

    raise last_err if last_err is not None else ValueError("Gemini image generation failed with no usable response")

def realize_single_layer_fullframe(image, layer_context, ordered_contexts, depth_fallback_masks, layer_guidance, output_dir, model_name):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    order = layer_context["order"]
    layer_name = layer_context["layer_name"]

    mask_pack = build_control_masks_for_layer(
        layer_context,
        ordered_contexts,
        depth_fallback_masks,
    )

    visible_mask = layer_context["visible_mask"] > 0
    focused_input = apply_focus_mask_fullframe(
        image_rgb=image,
        show_mask=mask_pack["show_mask"],
        black_mask=mask_pack["black_mask"],
        beige_value=235,
    )

    prompt = build_fullframe_prompt(layer_context, layer_guidance)
    generated_full = gemini_edit(
        focused_input,
        prompt=prompt,
        model_name=model_name,
    )

    target_h, target_w = focused_input.shape[:2]
    if generated_full.shape[:2] != (target_h, target_w):
        generated_full = np.array(
            Image.fromarray(generated_full.astype(np.uint8)).resize(
                (target_w, target_h),
                Image.Resampling.LANCZOS,
            )
        )

    generated_full_path = output_dir / f"{order:02d}_{layer_name}_generated_full.png"
    generated_visible_full_path = output_dir / f"{order:02d}_{layer_name}_generated_visible_full.png"
    focused_input_path = output_dir / f"{order:02d}_{layer_name}_focused_input.png"

    Image.fromarray(focused_input).save(focused_input_path)
    Image.fromarray(generated_full).save(generated_full_path)
    Image.fromarray(mask_to_rgba_fullframe(generated_full, visible_mask)).save(generated_visible_full_path)

    return {
        "layer_name": layer_name,
        "order": order,
        "focused_input_path": str(focused_input_path),
        "generated_full_path": str(generated_full_path),
        "generated_visible_full_path": str(generated_visible_full_path),
    }

In [ ]:

# =========================
# PATCH: GENERALIZATION STABILIZERS
# =========================

# These overrides keep the AI-driven scene analysis, but constrain the parts
# that were drifting too far from the stable scene2 behavior.

CANONICAL_QUERY_SPECS = [
    {"query": "sky",      "synonyms": ["sky", "cloud", "clouds", "sunset", "sunrise"], "family": "sky_background"},
    {"query": "mountain", "synonyms": ["mountain", "mountains", "hill", "hills", "cliff"], "family": "distant_landscape"},
    {"query": "water",    "synonyms": ["water", "river", "lake", "sea", "ocean", "pond", "stream"], "family": "water"},
    {"query": "tree",     "synonyms": ["tree", "trees", "forest", "branch", "branches", "foliage", "bush", "bushes", "plant"], "family": "vegetation"},
    {"query": "building", "synonyms": ["building", "temple", "pagoda", "house", "church", "tower", "shrine", "gate", "bridge"], "family": "architecture"},
    {"query": "person",   "synonyms": ["person", "people", "man", "woman", "boy", "girl", "human", "tourist"], "family": "human_subject"},
    {"query": "animal",   "synonyms": ["animal", "deer", "dog", "cat", "horse", "bird", "cow", "sheep"], "family": "animal_subject"},
    {"query": "vehicle",  "synonyms": ["car", "truck", "bus", "train", "bicycle", "bike", "boat"], "family": "vehicle"},
    {"query": "flower",   "synonyms": ["flower", "flowers", "blossom", "petal"], "family": "vegetation_detail"},
    {"query": "statue",   "synonyms": ["statue", "sign", "bench", "lamp", "lantern"], "family": "manmade_object"},
]

DETAIL_BLACKLIST = {
    "path", "road", "ground", "grass", "bag", "shoulder bag", "cloud", "clouds",
    "distant people", "park path", "dark foliage", "tree branches", "dark tree branches on left",
    "left tree", "right tree", "foreground grass", "field", "lawn"
}

LABEL_CANONICAL_MAP = {
    "people": "person",
    "man": "person",
    "woman": "person",
    "boy": "person",
    "girl": "person",
    "human": "person",
    "tourist": "person",
    "deer": "animal",
    "dog": "animal",
    "cat": "animal",
    "horse": "animal",
    "bird": "animal",
    "cow": "animal",
    "sheep": "animal",
    "tree": "trees",
    "trees": "trees",
    "distantrees": "trees",
    "forest": "trees",
    "branch": "trees",
    "branches": "trees",
    "foliage": "trees",
    "bush": "trees",
    "bushes": "trees",
    "plant": "trees",
    "building": "building",
    "temple": "building",
    "pagoda": "building",
    "church": "building",
    "house": "building",
    "tower": "building",
    "shrine": "building",
    "gate": "building",
    "bridge": "building",
    "cloud": "sky",
    "clouds": "sky",
    "mountains": "mountain",
    "hill": "mountain",
    "hills": "mountain",
    "grass": "ground",
    "path": "ground",
    "road": "ground",
    "field": "ground",
    "lawn": "ground",
    "bag": "detail",
}

DROP_LABELS = {"ground", "detail"}

def canonicalize_label(label):
    l = clean_query_text(label)
    return LABEL_CANONICAL_MAP.get(l, l)

def normalized_family_from_label(label):
    l = canonicalize_label(label)
    if l == "sky":
        return "sky_background"
    if l == "mountain":
        return "distant_landscape"
    if l in {"building"}:
        return "architecture"
    if l in {"trees", "flower"}:
        return "vegetation"
    if l == "person":
        return "human_subject"
    if l == "animal":
        return "animal_subject"
    if l == "vehicle":
        return "manmade_object"
    return normalize_label_family(l)

def generate_queries(scene_analysis, fallback_caption="", max_queries=10):
    source_terms = []
    if isinstance(scene_analysis, dict):
        for key in [
            "candidate_detection_queries",
            "important_objects",
            "background_elements",
            "foreground_elements",
            "framing_elements",
            "notes_for_layering",
        ]:
            vals = scene_analysis.get(key, [])
            if isinstance(vals, str):
                vals = [vals]
            source_terms.extend(vals)

    source_terms.append(fallback_caption or "")
    source_blob = " | ".join(clean_query_text(x) for x in source_terms if x).lower()

    selected = []
    for spec in CANONICAL_QUERY_SPECS:
        if any(s in source_blob for s in spec["synonyms"]):
            selected.append(spec["query"])

    # Always anchor the composition with broad scene queries.
    if "sky" not in selected:
        selected.insert(0, "sky")
    if "person" not in selected and re.search(r"\b(man|woman|boy|girl|person|people|human)\b", source_blob):
        selected.append("person")
    if "animal" not in selected and re.search(r"\b(deer|dog|cat|horse|bird|animal)\b", source_blob):
        selected.append("animal")
    if "tree" not in selected:
        selected.append("tree")
    if "building" not in selected and re.search(r"\b(temple|pagoda|building|house|church|tower|shrine|gate|bridge)\b", source_blob):
        selected.append("building")

    # keep stable order, remove detail/ground queries
    selected = [q for q in dedupe_keep_order(selected) if q not in DETAIL_BLACKLIST]

    # conservative upper bound to keep Florence stable
    return selected[:max_queries]

def dedupe_boxes(boxes, image_shape, iou_threshold=0.75):
    clean = []
    H, W = image_shape[:2]
    img_area = float(H * W)

    for b in boxes:
        label = canonicalize_label(b.get("label", "object"))
        if label in DROP_LABELS:
            continue

        bbox = clamp_box(b["bbox"], image_shape)
        x1, y1, x2, y2 = bbox
        area = max(1, (x2 - x1) * (y2 - y1))
        area_ratio = area / img_area

        # remove tiny noise
        if area_ratio < 0.002:
            continue

        # keep sky only when it is really a broad backdrop
        if label == "sky" and area_ratio < 0.10:
            continue

        keep = True
        replace_idx = None
        for i, prev in enumerate(clean):
            prev_label = prev["label"]
            prev_bbox = prev["bbox"]
            iou = box_iou(prev_bbox, bbox)

            if prev_label == label and iou >= iou_threshold:
                # keep the larger of near-duplicate boxes
                prev_area = max(1, (prev_bbox[2]-prev_bbox[0])*(prev_bbox[3]-prev_bbox[1]))
                if area > prev_area:
                    replace_idx = i
                keep = False
                break

            # same-family containment cleanup
            if prev_label == label and (box_contains(prev_bbox, bbox) or box_contains(bbox, prev_bbox)):
                prev_area = max(1, (prev_bbox[2]-prev_bbox[0])*(prev_bbox[3]-prev_bbox[1]))
                if area > prev_area:
                    replace_idx = i
                keep = False
                break

        if replace_idx is not None:
            clean[replace_idx] = {"label": label, "bbox": bbox}
        elif keep:
            clean.append({"label": label, "bbox": bbox})

    # prefer one broad vegetation box over many fragmented ones
    trees = [b for b in clean if b["label"] == "trees"]
    if len(trees) > 1:
        x1 = min(b["bbox"][0] for b in trees)
        y1 = min(b["bbox"][1] for b in trees)
        x2 = max(b["bbox"][2] for b in trees)
        y2 = max(b["bbox"][3] for b in trees)
        clean = [b for b in clean if b["label"] != "trees"]
        clean.append({"label": "trees", "bbox": [x1, y1, x2, y2]})

    return clean

def box_contains(boxA, boxB, margin=8):
    ax1, ay1, ax2, ay2 = boxA
    bx1, by1, bx2, by2 = boxB
    return (ax1 - margin <= bx1 <= bx2 <= ax2 + margin) and (ay1 - margin <= by1 <= by2 <= ay2 + margin)

def postprocess_stable_objects(stable_objects):
    processed = []
    for obj in stable_objects:
        label = canonicalize_label(obj["label"])
        if label in DROP_LABELS:
            continue
        new_obj = dict(obj)
        new_obj["label"] = label
        new_obj["family"] = normalized_family_from_label(label)
        processed.append(new_obj)

    # merge same canonical label objects more aggressively
    merged = []
    for obj in processed:
        merged_to_existing = False
        for prev in merged:
            same_label = prev["label"] == obj["label"]
            overlap = np.logical_and(prev["mask"] > 0, obj["mask"] > 0).sum()
            smaller = min((prev["mask"] > 0).sum(), (obj["mask"] > 0).sum())
            if same_label and smaller > 0 and overlap / smaller > 0.35:
                prev["mask"] = np.logical_or(prev["mask"] > 0, obj["mask"] > 0)
                prev["bbox"] = [
                    min(prev["bbox"][0], obj["bbox"][0]),
                    min(prev["bbox"][1], obj["bbox"][1]),
                    max(prev["bbox"][2], obj["bbox"][2]),
                    max(prev["bbox"][3], obj["bbox"][3]),
                ]
                merged_to_existing = True
                break
        if not merged_to_existing:
            merged.append(obj)
    return merged

def reorder_layer_contexts_by_depth(layer_contexts, depth_map):
    def role_rank(ctx):
        fam = ctx.get("family", "object")
        if fam == "sky_background":
            return 0
        if fam == "distant_landscape":
            return 1
        if fam == "architecture":
            return 2
        if fam == "vegetation":
            return 3
        if fam == "animal_subject":
            return 4
        if fam == "human_subject":
            return 5
        return 6

    enriched = []
    for ctx in layer_contexts:
        ownership = ctx["ownership_mask"] > 0
        vals = depth_map[ownership]
        rep_depth = float(np.median(vals)) if vals.size else float(np.median(depth_map))
        new_ctx = dict(ctx)
        new_ctx["rep_depth"] = rep_depth
        enriched.append(new_ctx)

    # Smaller depth value tended to mean closer in your scene, so sort far->near by descending depth,
    # then force clear semantic anchor layers to remain stable.
    enriched = sorted(enriched, key=lambda c: (role_rank(c), -c["rep_depth"]))

    for i, ctx in enumerate(enriched):
        ctx["order"] = i
        ctx["front_layer_names"] = [x["layer_name"] for x in enriched[i+1:]]

    name_to_ctx = {ctx["layer_name"]: ctx for ctx in enriched}
    for ctx in enriched:
        front_labels = []
        for nm in ctx.get("front_layer_names", []):
            front_labels.extend(name_to_ctx[nm].get("labels", []))
        ctx["front_layer_labels"] = dedupe_keep_order([clean_query_text(x) for x in front_labels if x])

    return enriched

def assign_leftover_pixels_by_depth(layer_contexts, depth_map):
    H, W = depth_map.shape
    layer_stats = compute_layer_depth_stats(layer_contexts, depth_map)
    unassigned = build_unassigned_pixel_mask(layer_contexts, (H, W))
    fallback_masks = {ctx["layer_name"]: np.zeros((H, W), dtype=bool) for ctx in layer_contexts}

    # Do not spread fallback pixels into person/animal/foreground detail layers.
    # Keep the scene2 behavior: let broad backdrop / scenery absorb most leftover context.
    allowed = []
    for ctx in layer_contexts:
        fam = ctx.get("family", "object")
        if fam in {"sky_background", "distant_landscape", "architecture", "vegetation"}:
            allowed.append(ctx["layer_name"])

    if not allowed:
        allowed = [ctx["layer_name"] for ctx in layer_contexts]

    layer_depths = np.array([layer_stats[name]["rep_depth"] for name in allowed], dtype=np.float32)
    ys, xs = np.where(unassigned)
    if len(ys) == 0:
        return fallback_masks, unassigned, layer_stats

    pixel_depths = depth_map[ys, xs].astype(np.float32)
    dist = np.abs(pixel_depths[:, None] - layer_depths[None, :])
    best_idx = np.argmin(dist, axis=1)
    for k, layer_idx in enumerate(best_idx):
        fallback_masks[allowed[layer_idx]][ys[k], xs[k]] = True
    return fallback_masks, unassigned, layer_stats

def restrict_fallback_masks_generalized(layer_contexts, depth_fallback_masks):
    restricted = {}
    for ctx in layer_contexts:
        layer_name = ctx["layer_name"]
        fallback_mask = depth_fallback_masks[layer_name] > 0
        ownership_mask = ctx["ownership_mask"] > 0
        fam = ctx.get("family", "object")
        h, w = ownership_mask.shape

        if fam in {"human_subject", "animal_subject", "manmade_object"}:
            # Keep subject masks tight. No giant ground swallowing.
            local_window = expand_mask(
                ownership_mask.astype(np.uint8),
                kernel_size=dynamic_kernel_from_mask(ownership_mask, min_k=21, max_k=101, frac=0.10)
            )
            fallback_mask = fallback_mask & local_window

        elif fam == "architecture":
            local_window = expand_mask(
                ownership_mask.astype(np.uint8),
                kernel_size=dynamic_kernel_from_mask(ownership_mask, min_k=31, max_k=151, frac=0.16)
            )
            fallback_mask = fallback_mask & local_window

        elif fam == "sky_background":
            yy = np.arange(h)[:, None]
            upper_zone = yy <= int(h * 0.55)
            fallback_mask = fallback_mask & upper_zone

        elif fam in {"distant_landscape", "vegetation"}:
            # broad enough to collect context, but still bounded
            local_window = expand_mask(
                ownership_mask.astype(np.uint8),
                kernel_size=dynamic_kernel_from_mask(ownership_mask, min_k=41, max_k=201, frac=0.22)
            )
            fallback_mask = fallback_mask & local_window

        restricted[layer_name] = fallback_mask
    return restricted

def build_control_masks_for_layer(layer_context, ordered_contexts, depth_fallback_masks):
    order = layer_context["order"]
    layer_name = layer_context["layer_name"]
    fam = layer_context.get("family", "object")
    visible_mask = layer_context["visible_mask"] > 0
    fallback_mask = depth_fallback_masks[layer_name] > 0

    # Subjects stay anchor-first. Broad scenery can expose fallback support so
    # assigned leftover pixels actually become visible in the focused input.
    if fam in {"human_subject", "animal_subject", "manmade_object"}:
        show_mask = visible_mask.copy() if visible_mask.sum() > 0 else fallback_mask.copy()
    else:
        show_mask = visible_mask | fallback_mask

    H, W = visible_mask.shape
    front_visible = np.zeros((H, W), dtype=bool)
    behind_support = np.zeros((H, W), dtype=bool)

    for ctx in ordered_contexts:
        if ctx["layer_name"] == layer_name:
            continue
        if ctx["order"] > order:
            front_visible |= (ctx["visible_mask"] > 0)
        elif ctx["order"] < order:
            behind_support |= get_layer_support_mask(ctx, depth_fallback_masks)

    # Backdrop layers should not inherit giant black ceilings from layers behind them.
    if fam in {"sky_background", "distant_landscape"}:
        black_mask = np.zeros((H, W), dtype=bool)
    else:
        black_mask = behind_support.copy()

    beige_mask = ~(show_mask | black_mask)

    return {
        "anchor_mask": visible_mask,
        "fallback_mask": fallback_mask,
        "current_support": visible_mask | fallback_mask,
        "front_visible": front_visible,
        "behind_support": behind_support,
        "show_mask": show_mask,
        "black_mask": black_mask,
        "beige_mask": beige_mask,
    }


RECOVERY_QUERY_MAP = {
    "person": ["young man", "young woman", "man", "woman", "person", "people"],
    "animal": ["deer", "animal", "dog", "cat", "horse", "bird"],
    "building": ["temple", "pagoda", "building", "house", "shrine", "gate", "bridge"],
    "trees": ["tree", "trees", "forest", "branches", "foliage"],
}

def gather_text_blob(scene_analysis, fallback_caption=""):
    terms = []
    if isinstance(scene_analysis, dict):
        for key in [
            "scene_summary",
            "candidate_detection_queries",
            "important_objects",
            "background_elements",
            "foreground_elements",
            "framing_elements",
            "notes_for_layering",
        ]:
            vals = scene_analysis.get(key, [])
            if isinstance(vals, str):
                vals = [vals]
            terms.extend(vals)
    terms.append(fallback_caption or "")
    return " | ".join(clean_query_text(x) for x in terms if x).lower()

def build_recovery_queries(scene_analysis, fallback_caption, missing_labels):
    text_blob = gather_text_blob(scene_analysis, fallback_caption)
    recovery = []
    for label in missing_labels:
        for q in RECOVERY_QUERY_MAP.get(label, [label]):
            q_clean = clean_query_text(q)
            if not q_clean:
                continue
            if q_clean in text_blob or label in {"person", "animal", "trees", "building"}:
                recovery.append(q_clean)
    return dedupe_keep_order(recovery)

def labels_present_from_boxes(boxes):
    return {canonicalize_label(b.get("label", "object")) for b in boxes}

def family_specific_query_bias(scene_analysis, fallback_caption=""):
    text_blob = gather_text_blob(scene_analysis, fallback_caption)
    extra = []
    if "deer" in text_blob:
        extra.append("deer")
    if re.search(r"\byoung man\b", text_blob):
        extra.append("young man")
    elif re.search(r"\bman\b", text_blob):
        extra.append("man")
    elif re.search(r"\bwoman\b", text_blob):
        extra.append("woman")
    if re.search(r"\bpagoda\b|\btemple\b|\bshrine\b", text_blob):
        extra.append("temple")
    return dedupe_keep_order(extra)


In [ ]:
# =========================
# LOAD IMAGE + MODELS
# =========================

image = load_image(image_path, max_side=cfg.image_max_side)

show_image(image, title=f"{SCENE_NAME} — input", figsize=(8, 12))
print("image shape:", image.shape)

florence = FlorenceParser(cfg.florence_model)
segmenter = SAM2Segmenter(cfg.sam2_config, cfg.sam2_checkpoint)
depth_runner = DepthRunner(cfg.depth_encoder)

In [ ]:
# =========================
# SCENE ANALYSIS
# =========================

caption = florence.get_dense_caption(image)
print("Florence caption:")
print(caption)

scene_analysis = {
    "scene_summary": caption,
    "style_tags": [],
    "important_objects": [],
    "framing_elements": [],
    "background_elements": [],
    "foreground_elements": [],
    "candidate_detection_queries": [],
    "likely_indoor_or_outdoor": "mixed",
    "notes_for_layering": [],
}

if USE_VISION_LLM and VISION_PROVIDER in {"gemini", "openai"}:
    try:
        scene_analysis = analyze_scene_with_llm(
            image_rgb=image,
            fallback_caption=caption,
            provider=VISION_PROVIDER,
        )
    except Exception as e:
        print("Vision LLM analysis failed. Falling back to Florence caption only.")
        print("Error:", repr(e))

with open(SCENE_ANALYSIS_PATH, "w", encoding="utf-8") as f:
    json.dump(scene_analysis, f, indent=2, ensure_ascii=False)

print("Scene analysis:")
print(json.dumps(scene_analysis, indent=2, ensure_ascii=False))

In [ ]:
# =========================
# DYNAMIC FLORENCE QUERIES
# =========================

base_queries = generate_queries(scene_analysis, fallback_caption=caption, max_queries=MAX_QUERIES)
biased_queries = family_specific_query_bias(scene_analysis, fallback_caption=caption)
queries = dedupe_keep_order(base_queries + biased_queries)

print("Generated queries:")
for i, q in enumerate(queries, 1):
    print(f"{i:02d}. {q}")

all_boxes = []
raw_detection_dump = {}

def run_detection_queries(query_list, stage_name="base"):
    stage_boxes = []
    for q in query_list:
        det_q = florence.get_open_vocab_detection(image, q)
        boxes_q = parse_florence_boxes(det_q, image.shape)
        raw_detection_dump[f"{stage_name}:{q}"] = {
            "raw": det_q,
            "parsed": boxes_q,
        }
        print(f"\n[{stage_name.upper()}] QUERY:", q)
        print("PARSED:", boxes_q)
        stage_boxes.extend(boxes_q)
    return stage_boxes

all_boxes.extend(run_detection_queries(queries, stage_name="base"))
boxes = dedupe_boxes(all_boxes, image.shape, iou_threshold=0.75)

present = labels_present_from_boxes(boxes)
wanted = set()
text_blob = gather_text_blob(scene_analysis, caption)
if re.search(r"\b(man|woman|boy|girl|person|people|human|young man|young woman)\b", text_blob):
    wanted.add("person")
if re.search(r"\b(deer|dog|cat|horse|bird|animal)\b", text_blob):
    wanted.add("animal")
if re.search(r"\b(temple|pagoda|building|house|church|tower|shrine|gate|bridge)\b", text_blob):
    wanted.add("building")
if re.search(r"\b(tree|trees|forest|branch|branches|foliage|bush)\b", text_blob):
    wanted.add("trees")

missing = [lbl for lbl in ["person", "animal", "building", "trees"] if lbl in wanted and lbl not in present]
recovery_queries = build_recovery_queries(scene_analysis, caption, missing)

# If animals are present, explicitly try deer as a second pass to catch multiple deer.
if "animal" in wanted and "deer" in text_blob and "deer" not in recovery_queries:
    recovery_queries.append("deer")

recovery_queries = [q for q in dedupe_keep_order(recovery_queries) if q not in queries]

if recovery_queries:
    print("\nRecovery queries:")
    for i, q in enumerate(recovery_queries, 1):
        print(f"{i:02d}. {q}")
    all_boxes.extend(run_detection_queries(recovery_queries, stage_name="recovery"))
    boxes = dedupe_boxes(all_boxes, image.shape, iou_threshold=0.72)

with open(QUERY_DEBUG_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "queries": queries,
        "recovery_queries": recovery_queries,
    }, f, indent=2, ensure_ascii=False)

print("\nFINAL DEDUPED BOXES:")
print(boxes)
print("num boxes:", len(boxes))

fig, ax = plt.subplots(figsize=(8, 12))
ax.imshow(image)
for b in boxes:
    x1, y1, x2, y2 = b["bbox"]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, max(0, y1 - 6), b["label"], color="yellow", fontsize=10, backgroundcolor="black")
ax.axis("off")
ax.set_title("Florence detections — deduped")
plt.show()

with open(DEBUG_DIR / "raw_detection_dump.json", "w", encoding="utf-8") as f:
    json.dump(raw_detection_dump, f, indent=2, ensure_ascii=False, default=str)


In [ ]:
# =========================
# SEGMENTATION + DEPTH
# =========================

segmented = segmenter.segment_boxes(image, boxes)
merged_segmented = merge_segmented_by_label(segmented)
depth = depth_runner.infer(image)
stable_objects = build_stable_merged_objects(merged_segmented, depth)
stable_objects = postprocess_stable_objects(stable_objects)

for i, obj in enumerate(stable_objects):
    obj["id"] = i
    obj["family"] = normalized_family_from_label(obj.get("label", "object"))

print("num segmented objects:", len(segmented))
print("num merged objects:", len(merged_segmented))
print("num stable objects:", len(stable_objects))

show_image(depth, title="Depth map", figsize=(8, 12), cmap="viridis")

fig, ax = plt.subplots(figsize=(8, 12))
ax.imshow(image)
for obj in stable_objects:
    x1, y1, x2, y2 = obj["bbox"]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, max(0, y1 - 6), f"{obj['id']}:{obj['label']} [{obj['family']}]", color="white", fontsize=9, backgroundcolor="black")
ax.axis("off")
ax.set_title("Stable objects")
plt.show()

if DEBUG_SHOW_EACH_OBJECT_MASK:
    for obj in stable_objects:
        mask = obj["mask"] > 0
        overlay = image.copy()
        overlay[mask] = (0.65 * overlay[mask] + 0.35 * np.array([255, 80, 80])).astype(np.uint8)
        show_image(overlay, title=f"Object {obj['id']} — {obj['label']} — {obj['family']}", figsize=(6, 9))


In [ ]:
# =========================
# SCENE REPRESENTATION + LAYER CONTEXTS
# =========================

scene_repr = build_scene_representation(
    image_path=image_path,
    image_shape=image.shape,
    caption=scene_analysis.get("scene_summary", caption),
    stable_objects=stable_objects,
)
scene_repr["scene_analysis"] = scene_analysis

save_scene_representation(scene_repr, SCENE_REPR_PATH)

layer_plan = plan_layers_deterministic(scene_repr)
with open(LAYER_PLAN_PATH, "w", encoding="utf-8") as f:
    json.dump(layer_plan, f, indent=2, ensure_ascii=False)

mask_records = []
for obj in stable_objects:
    cleaned_mask = cleanup_mask(
        heuristic_complete(obj["mask"], obj["label"]),
        cfg.min_component_area,
        cfg.smooth_kernel,
    )
    out_mask = MASK_DIR / f"{SCENE_NAME}_mask_{obj['id']}.png"
    save_mask(out_mask, cleaned_mask)
    mask_records.append({
        "id": obj["id"],
        "label": obj["label"],
        "bbox": obj["bbox"],
        "mask_path": str(out_mask),
        "family": obj["family"],
    })

object_mask_map = build_object_mask_map(mask_records)

layer_contexts = build_layer_contexts(
    scene_repr=scene_repr,
    layer_plan=layer_plan,
    object_mask_map=object_mask_map,
)

object_id_to_family = {obj["id"]: obj["family"] for obj in stable_objects}

for ctx in layer_contexts:
    labels = [canonicalize_label(x) for x in ctx.get("labels", []) if canonicalize_label(x) not in DROP_LABELS]
    ctx["labels"] = dedupe_keep_order(labels)
    obj_families = [object_id_to_family.get(obj_id, normalized_family_from_label(lbl)) for obj_id, lbl in zip(ctx.get("object_ids", []), ctx["labels"])]
    if obj_families:
        family = obj_families[0]
    else:
        family = normalized_family_from_label(ctx["labels"][0] if ctx["labels"] else ctx["layer_name"])
    ctx["family"] = family
    ctx["scene_caption"] = scene_analysis.get("scene_summary", caption)

layer_contexts = [ctx for ctx in layer_contexts if (ctx["ownership_mask"] > 0).sum() > 0]
layer_contexts = reorder_layer_contexts_by_depth(layer_contexts, depth)

layer_df = summarize_layer_contexts(layer_contexts)
layer_df


In [ ]:
# =========================
# DEPTH FALLBACK + LAYER GUIDANCE
# =========================


raw_depth_fallback_masks, unassigned_pixels_mask, layer_depth_stats, leftover_assignment_index = assign_leftover_pixels_by_depth(layer_contexts, depth)
restricted_depth_fallback_masks = restrict_fallback_masks_generalized(layer_contexts, raw_depth_fallback_masks)

if USE_RAW_DEPTH_FALLBACK_FOR_GEMINI:
    depth_fallback_masks = raw_depth_fallback_masks
    print("Using RAW closest-depth leftover assignment for Gemini masks.")
else:
    depth_fallback_masks = restricted_depth_fallback_masks
    print("Using RESTRICTED leftover assignment for Gemini masks.")

show_image(unassigned_pixels_mask, title="Unassigned pixels after ownership masks", figsize=(8, 12), cmap="gray")
summarize_leftover_assignment(
    layer_contexts,
    raw_depth_fallback_masks,
    restricted_depth_fallback_masks,
    unassigned_pixels_mask,
    leftover_assignment_index,
)

if DEBUG_SHOW_LEFTOVER_ASSIGNMENT:
    show_leftover_assignment_map(
        layer_contexts,
        image,
        unassigned_pixels_mask,
        leftover_assignment_index,
    )

print("Depth-aware layer order:")
for ctx in layer_contexts:
    print({
        "layer_name": ctx["layer_name"],
        "family": ctx.get("family"),
        "order": ctx["order"],
        "rep_depth": round(ctx.get("rep_depth", -1), 6),
    })

layer_guidance_map = {}
for ctx in sorted(layer_contexts, key=lambda x: x["order"]):
    if USE_LLM_LAYER_GUIDANCE and VISION_PROVIDER in {"gemini", "openai"}:
        try:
            guidance = get_layer_guidance_with_llm(ctx, scene_analysis, provider=VISION_PROVIDER)
        except Exception as e:
            print(f"Layer guidance LLM failed for {ctx['layer_name']}. Falling back.")
            print("Error:", repr(e))
            guidance = build_layer_guidance_fallback(ctx, scene_analysis)
    else:
        guidance = build_layer_guidance_fallback(ctx, scene_analysis)

    layer_guidance_map[ctx["layer_name"]] = guidance

with open(LAYER_GUIDANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(layer_guidance_map, f, indent=2, ensure_ascii=False)

print(json.dumps(layer_guidance_map, indent=2, ensure_ascii=False))


In [ ]:
# =========================
# DEBUG LAYER CONTROL MASKS
# =========================

ordered_contexts = sorted(layer_contexts, key=lambda x: x["order"])

for ctx in ordered_contexts:
    layer_name = ctx["layer_name"]
    mask_pack = build_control_masks_for_layer(
        ctx,
        ordered_contexts,
        depth_fallback_masks,
    )
    focused_input = apply_focus_mask_fullframe(
        image_rgb=image,
        show_mask=mask_pack["show_mask"],
        black_mask=mask_pack["black_mask"],
        beige_value=235,
    )

    print("=" * 70)
    print("LAYER:", layer_name)
    print("family:", ctx.get("family"))
    print("labels:", ctx.get("labels"))
    print("front_layer_labels:", ctx.get("front_layer_labels"))
    print("anchor sum:", int(mask_pack["anchor_mask"].sum()))
    print("fallback sum:", int(mask_pack["fallback_mask"].sum()))
    print("current_support sum:", int(mask_pack["current_support"].sum()))
    print("front_visible sum:", int(mask_pack["front_visible"].sum()))
    print("behind_support sum:", int(mask_pack["behind_support"].sum()))
    print("show_mask sum:", int(mask_pack["show_mask"].sum()))
    print("black_mask sum:", int(mask_pack["black_mask"].sum()))
    print("beige_mask sum:", int(mask_pack["beige_mask"].sum()))
    print("show ∩ black:", int((mask_pack["show_mask"] & mask_pack["black_mask"]).sum()))
    print("anchor ∩ fallback:", int((mask_pack["anchor_mask"] & mask_pack["fallback_mask"]).sum()))

    if DEBUG_SHOW_EACH_LAYER_MASK:
        show_mask_debug(
            image_rgb=image,
            layer_name=layer_name,
            anchor_mask=mask_pack["anchor_mask"],
            fallback_mask=mask_pack["fallback_mask"],
            current_support=mask_pack["current_support"],
            front_visible=mask_pack["front_visible"],
            behind_support=mask_pack["behind_support"],
            show_mask=mask_pack["show_mask"],
            black_mask=mask_pack["black_mask"],
            focused_input=focused_input,
        )

In [ ]:
# =========================
# OPTIONAL: PREVIEW THE PROMPTS
# =========================

for ctx in ordered_contexts:
    prompt_preview = build_fullframe_prompt(
        layer_context=ctx,
        layer_guidance=layer_guidance_map[ctx["layer_name"]],
    )
    print("\n" + "=" * 80)
    print(ctx["layer_name"])
    print("=" * 80)
    print(prompt_preview[:2500])

In [ ]:
# =========================
# FULL-FRAME GENERATION
# =========================

if IMAGE_GENERATION_PROVIDER != "gemini":
    raise NotImplementedError("This notebook currently realizes the final full-frame layers with Gemini image generation.")

fullframe_results = []
failed_layers = []

for ctx in ordered_contexts:
    print("\n==============================")
    print("Processing full-frame layer:", ctx["layer_name"])
    print("==============================")

    try:
        result = realize_single_layer_fullframe(
            image=image,
            layer_context=ctx,
            ordered_contexts=ordered_contexts,
            depth_fallback_masks=depth_fallback_masks,
            layer_guidance=layer_guidance_map[ctx["layer_name"]],
            output_dir=FULLFRAME_OUTPUT_DIR,
            model_name=GEMINI_IMAGE_MODEL,
        )
        fullframe_results.append(result)
        print("Saved:")
        print("  focused_input      :", result["focused_input_path"])
        print("  generated_full     :", result["generated_full_path"])
        print("  generated_visible  :", result["generated_visible_full_path"])

    except Exception as e:
        failed_layers.append({"layer_name": ctx["layer_name"], "error": repr(e)})
        print(f"FAILED layer {ctx['layer_name']}: {e}")

print("\nCompleted layers:", len(fullframe_results))
print("Failed layers:", len(failed_layers))
if failed_layers:
    print(json.dumps(failed_layers, indent=2))

In [ ]:
# =========================
# VISUALIZE RESULTS
# =========================

for result in fullframe_results:
    print(result["layer_name"])
    for name, p in [
        ("focused_input", result["focused_input_path"]),
        ("generated_full", result["generated_full_path"]),
        ("generated_visible_full", result["generated_visible_full_path"]),
    ]:
        img = Image.open(p)
        plt.figure(figsize=(8, 10))
        plt.imshow(img)
        plt.title(f'{result["layer_name"]} — {name}')
        plt.axis("off")
        plt.show()

In [ ]:
# =========================
# ZIP OUTPUTS
# =========================

output_zip_base = Path("/content") / f"{SCENE_NAME}_generalized_fullframe_outputs"
shutil.make_archive(str(output_zip_base), "zip", str(SCENE_DIR))

print(f"Compressed '{SCENE_DIR}' to '{output_zip_base}.zip'")
print("Download path:", f"{output_zip_base}.zip")